# Week 5

# PySpark Data Cleaning, Transformation and Aggregation

## Celebal Technologies Data Engineering Internship

### Objective

The objective of this assignment is to understand Apache Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, and schema modification using PySpark DataFrames.

## Import Required Libraries

In [2]:
import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql.functions import *

from pyspark.sql.types import *

## Create Spark Session

In [3]:
spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


## Load Dataset

In [4]:
pdf = pd.read_csv(
    "data/Sample - Superstore.csv",
    encoding="latin1"
)

df = spark.createDataFrame(pdf)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


## Display Dataset

In [5]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Print Schema

In [6]:
df.printSchema()

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



## Display Columns

In [7]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Data Preparation

In [8]:
from pyspark.sql.functions import lit

df = (
    df
    .withColumn("status", lit(None).cast("string"))
    .withColumn("subscription", lit("Premium"))
    .withColumn("email", lit("sample@gmail.com"))
    .withColumn("store_id", lit("Store_01"))
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------+------------+----------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|status|subscription|           email|store_id|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------+------------+----------------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Cla

# Question 1

## What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional MapReduce has several limitations that make it less efficient for modern big data processing.

- It stores intermediate results on disk after every processing stage, which increases execution time.
- It is inefficient for iterative algorithms such as Machine Learning because data must be read from disk repeatedly.
- It has high latency for interactive data analysis.
- Programming with separate Map and Reduce functions increases development complexity.
- It does not provide built-in support for SQL, Machine Learning, Graph Processing, or Streaming.

Apache Spark overcomes these limitations by using in-memory computation, providing significantly faster execution, lower latency, simplified APIs, and integrated libraries for SQL, Machine Learning, Graph Processing, and Streaming.

# Question 2

## Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

### Answer

Apache Spark stores intermediate processing data in RAM instead of writing it to disk after every operation.

When an iterative Machine Learning algorithm performs multiple iterations, Spark reuses the cached data directly from memory, eliminating repeated disk I/O operations.

This results in significantly faster execution, making Spark ideal for algorithms such as K-Means, Logistic Regression, Decision Trees, and Linear Regression.

Compared to Hadoop MapReduce, Spark can execute iterative workloads several times faster because the data remains in memory throughout the computation.

# Question 3

## Remove duplicate rows based on Customer ID and Order Date.

### Answer

Duplicate records can lead to incorrect analysis and aggregation results.

Spark provides the `dropDuplicates()` function to remove duplicate rows based on selected columns.

In this dataset, **Customer ID** is used instead of **user_id** and **Order Date** is used instead of **transaction_date**.

In [9]:
print("Original Number of Records :", df.count())

df_q3 = df.dropDuplicates(["Customer ID", "Order Date"])

print("Records After Removing Duplicates :", df_q3.count())

df_q3.select(
    "Customer ID",
    "Order Date",
    "Sales"
).show(10)

Original Number of Records : 9994
Records After Removing Duplicates : 4992
+-----------+----------+-------+
|Customer ID|Order Date|  Sales|
+-----------+----------+-------+
|   DR-12880|11/28/2015| 12.132|
|   JS-15940| 1/17/2015|254.744|
|   DB-13210| 3/22/2015| 18.392|
|   JF-15490| 12/7/2015|   3.96|
|   PK-19075|  9/5/2016|  12.22|
|   LF-17185|10/20/2016|  7.152|
|   DK-13150| 12/5/2014| 24.816|
|   KC-16540| 9/11/2016|   7.61|
|   CR-12625|  6/1/2014|  45.48|
|   FA-14230| 9/19/2014|   7.16|
+-----------+----------+-------+
only showing top 10 rows


# Question 4

## Filter records where Region is West and calculate the average Sales for each Category.

### Answer

The dataset is filtered to include only records belonging to the **West** region.

The filtered data is grouped by **Category**, and the average Sales value is calculated using the `avg()` aggregation function.

In [10]:
from pyspark.sql.functions import avg, col

df_q4 = (
    df.filter(col("Region") == "West")
      .groupBy("Category")
      .agg(
          avg("Sales").alias("Average Sales")
      )
)

df_q4.show()

+---------------+------------------+
|       Category|     Average Sales|
+---------------+------------------+
|Office Supplies|116.42237691091192|
|      Furniture| 357.3023246110325|
|     Technology|420.68753255425696|
+---------------+------------------+



# Question 5

## Explain the difference between `.na.drop()` and `.na.fill()`. Fill the status column with 'Unknown'.

### Answer

Spark provides two methods for handling missing values.

- **na.drop()** removes rows containing null values.
- **na.fill()** replaces null values with a specified value while preserving the remaining data.

Replacing null values is generally preferred when the missing information should not result in data loss.

In [11]:
df_q5 = df.na.fill(
    {"status": "Unknown"}
)

df_q5.select("status").show(10, truncate=False)

+-------+
|status |
+-------+
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
+-------+
only showing top 10 rows


# Question 6

## Find the total count of records for each city where the count is greater than 100.

### Answer

The dataset is grouped by **City**, and the total number of records is calculated using the `count()` aggregation function. Only cities having more than **100 records** are displayed using the `filter()` condition.

In [13]:
from pyspark.sql.functions import count, col

df_q6 = (
    df.groupBy("City")
      .agg(count("*").alias("Total Records"))
      .filter(col("Total Records") > 100)
)

df_q6.show()

+-------------+-------------+
|         City|Total Records|
+-------------+-------------+
|  Springfield|          163|
|       Dallas|          157|
| Philadelphia|          537|
|  Los Angeles|          747|
|San Francisco|          510|
|    San Diego|          170|
|      Detroit|          115|
|     Columbus|          222|
|      Chicago|          314|
|      Seattle|          428|
|New York City|          915|
|      Houston|          377|
| Jacksonville|          125|
+-------------+-------------+



# Question 7

## How does the immutability of Spark DataFrames affect data cleaning operations?

### Answer

Spark DataFrames are **immutable**, which means their data cannot be modified after creation.

Whenever a data cleaning operation such as dropping columns, renaming columns, or replacing null values is performed, Spark creates a **new DataFrame** instead of modifying the original one.

This approach improves fault tolerance, enables parallel processing, and allows Spark to optimize execution through its query planner.

# Question 8

## Filter rows where Quantity is between 2 and 5 (inclusive) and Subscription is 'Premium'.

Note: The assignment mentions age, but my dataset doesn't have an age column. I am  using Quantity to demonstrate the same filtering concept.


### Answer

The provided dataset does not contain an **Age** column. Therefore, the **Quantity** column is used to demonstrate filtering between two values while also checking whether the subscription type is **Premium**.

In [14]:
df_q8 = df.filter(
    (col("Quantity").between(2, 5)) &
    (col("subscription") == "Premium")
)

df_q8.select(
    "Customer Name",
    "Quantity",
    "subscription",
    "Sales"
).show(10)

+---------------+--------+------------+--------+
|  Customer Name|Quantity|subscription|   Sales|
+---------------+--------+------------+--------+
|    Claire Gute|       2|     Premium|  261.96|
|    Claire Gute|       3|     Premium|  731.94|
|Darrin Van Huff|       2|     Premium|   14.62|
| Sean O'Donnell|       5|     Premium|957.5775|
| Sean O'Donnell|       2|     Premium|  22.368|
|Brosina Hoffman|       4|     Premium|    7.28|
|Brosina Hoffman|       3|     Premium|  18.504|
|Brosina Hoffman|       5|     Premium|   114.9|
|Brosina Hoffman|       4|     Premium| 911.424|
|   Andrew Allen|       3|     Premium|  15.552|
+---------------+--------+------------+--------+
only showing top 10 rows


# Question 9

## Why should null values be handled before performing mathematical aggregations?

### Answer

Handling null values before aggregation is important because missing values can produce incorrect analytical results.

Replacing or removing null values ensures that functions such as **sum()**, **avg()**, **min()**, and **max()** return meaningful and reliable outputs.

Proper data cleaning also improves overall data quality and prevents unexpected errors during processing.

# Question 10

## Cast the Order Date column to DateType and rename it as event_time.

Note: The assignment mentions raw_timestamp, but my dataset has Order Date.

### Answer

The provided dataset does not contain a **raw_timestamp** column.

Therefore, the **Order Date** column is converted to **DateType** and renamed as **event_time** to demonstrate schema modification.


In [15]:
from pyspark.sql.functions import to_date

df_q10 = (
    df.withColumn(
        "event_time",
        to_date(col("Order Date"), "M/d/yyyy")
    )
)

df_q10.select(
    "Order Date",
    "event_time"
).show(10)

+----------+----------+
|Order Date|event_time|
+----------+----------+
| 11/8/2016|2016-11-08|
| 11/8/2016|2016-11-08|
| 6/12/2016|2016-06-12|
|10/11/2015|2015-10-11|
|10/11/2015|2015-10-11|
|  6/9/2014|2014-06-09|
|  6/9/2014|2014-06-09|
|  6/9/2014|2014-06-09|
|  6/9/2014|2014-06-09|
|  6/9/2014|2014-06-09|
+----------+----------+
only showing top 10 rows


# Question 11

## Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

### Answer

A shuffle is the process of redistributing data across different partitions in a Spark cluster.

Operations such as **groupBy()**, **join()**, and **distinct()** require data from multiple partitions to be exchanged between executors.

This movement of data across partitions is called a **shuffle**.

Since records from different partitions are combined, grouping operations are classified as **wide transformations**. Shuffle operations are expensive because they involve network communication and disk I/O.

# Question 12

## Remove rows where the email column contains null values OR the Customer Name is empty.

### Answer

The assignment originally mentions **username**, but this dataset contains **Customer Name** instead. Therefore, Customer Name is used to demonstrate the same data-cleaning concept.

In [16]:
from pyspark.sql.functions import col, trim

df_q12 = df.filter(
    (col("email").isNotNull()) &
    (trim(col("Customer Name")) != "")
)

df_q12.select(
    "Customer Name",
    "email"
).show(10, truncate=False)

+---------------+----------------+
|Customer Name  |email           |
+---------------+----------------+
|Claire Gute    |sample@gmail.com|
|Claire Gute    |sample@gmail.com|
|Darrin Van Huff|sample@gmail.com|
|Sean O'Donnell |sample@gmail.com|
|Sean O'Donnell |sample@gmail.com|
|Brosina Hoffman|sample@gmail.com|
|Brosina Hoffman|sample@gmail.com|
|Brosina Hoffman|sample@gmail.com|
|Brosina Hoffman|sample@gmail.com|
|Brosina Hoffman|sample@gmail.com|
+---------------+----------------+
only showing top 10 rows


# Question 13

## Calculate the minimum, maximum and average Sales using the agg() function.

### Answer

The `agg()` function allows multiple aggregate calculations to be performed in a single operation. Here, the minimum, maximum and average values of the Sales column are calculated together.

In [17]:
from pyspark.sql.functions import min, max, avg

df_q13 = df.agg(
    min("Sales").alias("Minimum Sales"),
    max("Sales").alias("Maximum Sales"),
    avg("Sales").alias("Average Sales")
)

df_q13.show()

+-------------+-------------+------------------+
|Minimum Sales|Maximum Sales|     Average Sales|
+-------------+-------------+------------------+
|        0.444|     22638.48|229.85800083049827|
+-------------+-------------+------------------+



# Question 14

## What is the risk of using inferSchema=True when source data contains inconsistent date formats?

### Answer

Using `inferSchema=True` on datasets containing inconsistent date formats may lead Spark to infer incorrect data types.

Some values may be interpreted as strings while others are parsed as dates, causing schema inconsistencies and processing errors.

For production environments, it is recommended to define the schema explicitly instead of relying entirely on schema inference.


# Question 15

## Final Processing Pipeline

### Answer

The provided dataset does not contain a **store_id** column. A sample `store_id` column was created during the data preparation step to demonstrate the required processing pipeline.

The pipeline performs the following operations:

- Removes duplicate records
- Replaces null values in Sales with 0
- Groups data by store_id
- Calculates the total revenue for each store

In [18]:
from pyspark.sql.functions import sum

df_q15 = (
    df.dropDuplicates()
      .na.fill({"Sales": 0})
      .groupBy("store_id")
      .agg(
          sum("Sales").alias("Total Revenue")
      )
)

df_q15.show()

+--------+-----------------+
|store_id|    Total Revenue|
+--------+-----------------+
|Store_01|2297200.860299997|
+--------+-----------------+



# Conclusion

In this assignment, Apache Spark DataFrames were used to perform data cleaning, transformation, filtering, aggregation, schema modification, and pipeline creation. Various Spark operations such as `dropDuplicates()`, `groupBy()`, `agg()`, `filter()`, `na.fill()`, and type casting were successfully demonstrated using the Superstore dataset. This assignment highlights how PySpark simplifies large-scale data processing through distributed computation and efficient DataFrame APIs.